# Step 8 — Mask-based Baselines (EoMT)
Evaluates MSP, MaxLogit, MaxEntropy, RbA on 3 checkpoints (Cityscapes, COCO, Finetuned).
Also evaluates Temperature Scaling on MSP.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

REPO_URL = "https://github.com/flaviofrasca/MaskArchitectureAnomaly_CourseProject.git"
REPO_DIR = "/content/MaskArchitectureAnomaly_CourseProject"
BRANCH   = "step_8"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}
    print("Repo aggiornato")

# eval_anomaly_eomt.py deve girare da eomt/ perche importa i moduli locali
%cd {REPO_DIR}/eomt

Cloning into '/content/MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 289, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 289 (delta 62), reused 52 (delta 48), pack-reused 155 (from 2)
Receiving objects: 100% (289/289), 27.72 MiB | 21.28 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [7]:
!pip install ood-metrics timm lightning transformers torchmetrics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 57.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version

## Checkpoint Cityscapes

In [ ]:
!python eval_anomaly_eomt.py --model_type cityscapes --method MSP
!python eval_anomaly_eomt.py --model_type cityscapes --method MaxLogit
!python eval_anomaly_eomt.py --model_type cityscapes --method MaxEntropy
!python eval_anomaly_eomt.py --model_type cityscapes --method RbA

## Checkpoint COCO

In [ ]:
!python eval_anomaly_eomt.py --model_type coco --method MSP
!python eval_anomaly_eomt.py --model_type coco --method MaxLogit
!python eval_anomaly_eomt.py --model_type coco --method MaxEntropy
!python eval_anomaly_eomt.py --model_type coco --method RbA

## Checkpoint Finetuned

In [ ]:
!python eval_anomaly_eomt.py --model_type finetuned --method MSP
!python eval_anomaly_eomt.py --model_type finetuned --method MaxLogit
!python eval_anomaly_eomt.py --model_type finetuned --method MaxEntropy
!python eval_anomaly_eomt.py --model_type finetuned --method RbA

## Temperature Scaling (MSP)

PRO TIP: il modello gira **una volta sola** per checkpoint e salva i logits su disco.
Le 4 temperature vengono valutate caricando i logits cached, senza rieseguire il forward pass.

**mIoU**: non disponibile sui dataset anomaly (nessuna GT semantica) — colonna N/A.

In [ ]:
# Cityscapes — run once: caches logits + evaluates all temperatures
!python eval_temperature_scaling.py --model_type cityscapes

In [ ]:
# COCO
# !python eval_temperature_scaling.py --model_type coco

In [8]:
# PATCH per il salvataggio delle immagini in cache (da 2048x1024 a 640x640)
filepath = "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py"
with open(filepath) as f:
    code = f.read()

OLD = """        H, W = pixel_logits.shape[1], pixel_logits.shape[2]
        if ood_gts.shape != (H, W):
            ood_gts = np.array(
                Image.fromarray(ood_gts).resize((W, H), Image.NEAREST)
            )

        np.save(logits_path, pixel_logits.cpu().numpy().astype(np.float16))
        np.save(gt_path, ood_gts)"""

NEW = """        # Resize to _RESIZE_TO at save time (~5x smaller cache)
        TH, TW = _RESIZE_TO
        pixel_logits = F.interpolate(
            pixel_logits.unsqueeze(0), size=(TH, TW), mode='bilinear', align_corners=False
        ).squeeze(0)
        ood_gts = np.array(Image.fromarray(ood_gts).resize((TW, TH), Image.NEAREST))

        np.save(logits_path, pixel_logits.cpu().numpy().astype(np.float16))
        np.save(gt_path, ood_gts)"""

if OLD in code:
    code = code.replace(OLD, NEW)
    with open(filepath, 'w') as f:
        f.write(code)
    print("Patch OK")
else:
    print("Pattern non trovato — verifica indentazione")


Pattern non trovato — verifica indentazione


In [9]:
# AUTOCLICKER
%%javascript
function keepAlive() {
    console.log("Keep-alive: " + new Date().toLocaleTimeString());
    window.scrollBy(0, 1);
    window.scrollBy(0, -1);
    setTimeout(keepAlive, 60000);
}
keepAlive();

<IPython.core.display.Javascript object>

In [10]:
#altri due PATCH
filepath = "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py"
with open(filepath) as f:
    code = f.read()

# Verifica stato attuale
has_cache_dir_arg = "'--cache_dir'" in code
has_cache_dir_use = "args.cache_dir if args.cache_dir" in code
print(f"--cache_dir argparse: {has_cache_dir_arg}")
print(f"cache_dir assignment: {has_cache_dir_use}")

if not has_cache_dir_arg:
    code = code.replace(
        "help='Re-run inference even if cache exists')",
        "help='Re-run inference even if cache exists')\n    parser.add_argument('--cache_dir', default=None,\n                        help='Directory per i logit cachati')"
    )

if not has_cache_dir_use:
    code = code.replace(
        'cache_dir = f"logits_cache_{args.model_type}"',
        'cache_dir = args.cache_dir if args.cache_dir else f"logits_cache_{args.model_type}"'
    )

with open(filepath, 'w') as f:
    f.write(code)
print("Patch OK")


--cache_dir argparse: True
cache_dir assignment: True
Patch OK


In [11]:
CACHE_DIR = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/logits_cache_coco"
!python eval_temperature_scaling.py --model_type coco --cache_dir {CACHE_DIR}



Using cached logits: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/logits_cache_coco/  (219 files)

Model: EoMT [coco]  |  Temperature Scaling on MSP
Loading logits into memory and evaluating all temperatures ...
  RA-21 ...
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/npyio.py", line 456, in load
    return format.read_array(fid, allow_pickle=allow_pickle,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/format.py", line 839, in read_array
    array.shape = shape
    ^^^^^^^^^^^
ValueError: cannot reshape array of size 581568 into shape (133,640,640)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py", line 295, in <module>
    main()
  File "/content/MaskArchitectureAnomaly_CourseProje

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

# Usa disco locale invece di Drive
LOCAL_CACHE = "/content/logits_cache_coco"

!python eval_temperature_scaling.py --model_type coco --cache_dir {LOCAL_CACHE} --force_cache


Mounted at /content/drive

Loading EoMT [coco] for logits caching ...
model.safetensors: 100% 346M/346M [00:02<00:00, 139MB/s]
Loaded weights: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/coco/eomt_coco.bin
  Caching: SMIYC RA-21 ...
    10 images saved.
  Caching: SMIYC RO-21 ...
    30 images saved.
  Caching: FS L&F ...
    99 images saved.
  Caching: FS Static ...
    20 images saved.
  Caching: Road Anomaly ...
    60 images saved.

Logits cached in: /content/logits_cache_coco/

Model: EoMT [coco]  |  Temperature Scaling on MSP
Loading logits into memory and evaluating all temperatures ...
  RA-21 ...
  RO-21 ...
  L&F ...


In [1]:
!sed -n '160,260p' /content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py


    """Compute AuPRC and FPR95 from lists of score maps and GT masks (handles variable sizes)."""
    val_out_parts, val_label_parts = [], []
    for score, gt in zip(all_scores, all_gts):
        ood = score[gt == 1]
        ind = score[gt == 0]
        val_out_parts.append(np.concatenate([ind, ood]))
        val_label_parts.append(np.concatenate([np.zeros(len(ind)), np.ones(len(ood))]))
    val_out   = np.concatenate(val_out_parts)
    val_label = np.concatenate(val_label_parts)
    return (average_precision_score(val_label, val_out) * 100.0,
            fpr_at_95_tpr(val_out, val_label) * 100.0)


def evaluate_temperatures_for_dataset(all_logits, all_gts, T_list):
    """Evaluate all temperatures in T_list using pre-loaded logits. Returns dict T -> (auprc, fpr95)."""
    if not all_logits:
        return {T: None for T in T_list}
    results = {}
    for T in T_list:
        scores = [msp_score(logits, T) for logits in all_logits]
        results[T] = metrics_from_scores(scores, all

In [2]:
filepath = "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py"
with open(filepath) as f:
    code = f.read()

# 1. Aggiungi la funzione streaming prima di main()
STREAMING_FUNC = '''
def evaluate_temperatures_streaming(cache_dir, dataset_name, T_list):
    """Streaming: carica un'immagine alla volta, calcola tutti i T per immagine."""
    slug = _slug(dataset_name)
    logits_files = sorted(glob.glob(os.path.join(cache_dir, f"{slug}_*_logits.npy")))
    if not logits_files:
        return {T: None for T in T_list}

    all_scores_per_T = {T: [] for T in T_list}
    all_gts = []

    for lp in logits_files:
        gp = lp.replace("_logits.npy", "_gt.npy")
        if not os.path.exists(gp):
            continue
        logit = np.load(lp).astype(np.float32)
        gt    = np.load(gp)
        H, W  = logit.shape[1], logit.shape[2]
        if (H, W) != _RESIZE_TO:
            t = torch.tensor(logit).unsqueeze(0)
            logit = F.interpolate(t, size=_RESIZE_TO, mode="bilinear",
                                  align_corners=False).squeeze(0).numpy()
            gt = np.array(Image.fromarray(gt).resize(
                (_RESIZE_TO[1], _RESIZE_TO[0]), Image.NEAREST))

        logit_t = torch.tensor(logit, dtype=torch.float32)
        for T in T_list:
            score = 1.0 - torch.softmax(logit_t / T, dim=0).max(dim=0).values
            all_scores_per_T[T].append(score.numpy())
        all_gts.append(gt)

    results = {}
    for T in T_list:
        results[T] = metrics_from_scores(all_scores_per_T[T], all_gts)
        del all_scores_per_T[T]
    return results

'''

code = code.replace("# ---------------------------------------------------------------------------\n# Main",
                    STREAMING_FUNC + "# ---------------------------------------------------------------------------\n# Main")

# 2. Sostituisci le due righe nel loop di main
code = code.replace(
    "        logits_list, gts_list = load_dataset_logits(cache_dir, name)\n        all_results[name] = evaluate_temperatures_for_dataset(logits_list, gts_list, T_all)",
    "        all_results[name] = evaluate_temperatures_streaming(cache_dir, name, T_all)"
)

with open(filepath, 'w') as f:
    f.write(code)

# Verifica
assert "evaluate_temperatures_streaming" in open(filepath).read()
assert "load_dataset_logits(cache_dir, name)" not in open(filepath).read()
print("Patch OK")


Patch OK


In [5]:
LOCAL_CACHE = "/content/logits_cache_coco"
!cd /content/MaskArchitectureAnomaly_CourseProject/eomt && python eval_temperature_scaling.py --model_type coco --cache_dir {LOCAL_CACHE}



Using cached logits: /content/logits_cache_coco/  (219 files)

Model: EoMT [coco]  |  Temperature Scaling on MSP
Loading logits into memory and evaluating all temperatures ...
  RA-21 ...
  RO-21 ...
  L&F ...
^C


In [6]:
filepath = "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py"
with open(filepath) as f:
    code = f.read()

code = code.replace(
    "        logit_t = torch.tensor(logit, dtype=torch.float32)",
    "        _dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n        logit_t = torch.tensor(logit, dtype=torch.float32).to(_dev)"
)
code = code.replace(
    "            score = 1.0 - torch.softmax(logit_t / T, dim=0).max(dim=0).values\n            all_scores_per_T[T].append(score.numpy())",
    "            score = 1.0 - torch.softmax(logit_t / T, dim=0).max(dim=0).values\n            all_scores_per_T[T].append(score.cpu().numpy())"
)
with open(filepath, 'w') as f:
    f.write(code)
print("Patch OK")


Patch OK


In [7]:
LOCAL_CACHE = "/content/logits_cache_coco"
!cd /content/MaskArchitectureAnomaly_CourseProject/eomt && python eval_temperature_scaling.py --model_type coco --cache_dir {LOCAL_CACHE}



Using cached logits: /content/logits_cache_coco/  (219 files)

Model: EoMT [coco]  |  Temperature Scaling on MSP
Loading logits into memory and evaluating all temperatures ...
  RA-21 ...
  RO-21 ...
  L&F ...
  Static ...
  RoadAnom ...

Method              mIoU     RA-21 AuPRC     RA-21 FPR95     RO-21 AuPRC     RO-21 FPR95       L&F AuPRC       L&F FPR95    Static AuPRC    Static FPR95  RoadAnom AuPRC  RoadAnom FPR95
---------------------------------------------------------------------------
MSP                  N/A           38.87           77.51            2.77           99.98            3.80           97.22            4.70           99.47           18.89           91.84
MSP(t=0.5)           N/A           38.59           78.21            2.75           99.98            3.79           97.23            4.69           99.49           18.83           91.99
MSP(t=0.75)          N/A           38.80           77.64            2.76           99.98            3.80           97.23         

In [ ]:
# Finetuned
!python eval_temperature_scaling.py --model_type finetuned

## Risultati

In [ ]:
REPO_DIR = "/content/MaskArchitectureAnomaly_CourseProject"
print("=== MSP / MaxLogit / MaxEntropy / RbA ===")
with open(f"{REPO_DIR}/eomt/results_eomt.txt") as f:
    print(f.read())

print("\n=== Temperature Scaling ===")
with open(f"{REPO_DIR}/eomt/results_temperature_scaling.txt") as f:
    print(f.read())